# STEP 5 — Analysis & Paper Figures

**What this step does:**
- Loads all result CSVs from steps 2–4
- Generates all 4 paper figures + a summary table
- Interprets results and flags whether the main hypothesis is confirmed

**Figures produced:**
- `figures/fig1_accuracy_scaling.png` — Accuracy vs model size, CoT vs Direct (3 datasets)
- `figures/fig2_ece_scaling.png` — ECE vs model size, CoT vs Direct
- `figures/fig3_reliability_diagrams.png` — Reliability diagrams for key models
- `figures/fig4_consistency_delta.png` — Consistency delta (CoT - Direct) vs model size
- `figures/fig5_combined_trustworthiness.png` — Combined heatmap: 3 axes × 7 models

**Prerequisite:** Steps 2, 3, 4 must all be complete

In [ ]:
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

sys.path.insert(0, '/kaggle/working/light_paper')

BASE_DIR    = Path('/kaggle/working/light_paper')
RESULTS_DIR = BASE_DIR / 'results'
FIGURES_DIR = BASE_DIR / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)

from src.models import MODEL_REGISTRY, MODEL_SIZES, MODEL_NAMES

# Plot style
plt.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'figure.dpi': 150,
    'lines.linewidth': 2,
    'lines.markersize': 7,
})

COLORS = {'direct': '#1f77b4', 'cot': '#d62728'}
MARKERS= {'direct': 'o',       'cot': 's'}
DATASETS = ['gsm8k', 'bbh', 'truthfulqa']
DATASET_LABELS = {'gsm8k': 'GSM8K', 'bbh': 'BIG-Bench Hard', 'truthfulqa': 'TruthfulQA'}

# Load results
acc_df  = pd.read_csv(RESULTS_DIR / 'accuracy_summary.csv')
ece_df  = pd.read_csv(RESULTS_DIR / 'calibration_ece.csv')
con_df  = pd.read_csv(RESULTS_DIR / 'consistency_summary.csv')
delta_df= pd.read_csv(RESULTS_DIR / 'consistency_delta.csv')

with open(RESULTS_DIR / 'calibration_bin_data.json') as f:
    bin_data = json.load(f)

# Add size_b to acc/ece/con via model registry
size_map = {m.name: m.size_b for m in MODEL_REGISTRY}
for df in [acc_df, ece_df, con_df, delta_df]:
    if 'size_b' not in df.columns:
        df['size_b'] = df['model'].map(size_map)

print('All results loaded.')

## Figure 1 — Accuracy vs Model Size

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)

for ax, ds in zip(axes, DATASETS):
    subset = acc_df[acc_df.dataset == ds].sort_values('size_b')
    for mode in ['direct', 'cot']:
        m_data = subset[subset.mode == mode]
        ax.plot(
            m_data['size_b'], m_data['accuracy'],
            color=COLORS[mode], marker=MARKERS[mode],
            label=mode.capitalize()
        )

    ax.set_title(DATASET_LABELS[ds])
    ax.set_xlabel('Model size (B params)')
    ax.set_ylabel('Accuracy')
    ax.set_ylim(0, 1)
    ax.axvline(x=3, color='gray', linestyle='--', alpha=0.4, label='3B threshold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle('Figure 1: Accuracy (Direct vs CoT) across Model Sizes', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig1_accuracy_scaling.png', bbox_inches='tight')
plt.show()
print('✓ fig1 saved')

## Figure 2 — ECE (Calibration) vs Model Size

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, ds in zip(axes, ['gsm8k', 'truthfulqa']):
    subset = ece_df[ece_df.dataset == ds].sort_values('size_b')
    for mode in ['direct', 'cot']:
        m_data = subset[subset.mode == mode]
        ax.plot(
            m_data['size_b'], m_data['ece'],
            color=COLORS[mode], marker=MARKERS[mode],
            label=mode.capitalize()
        )

    ax.set_title(DATASET_LABELS[ds])
    ax.set_xlabel('Model size (B params)')
    ax.set_ylabel('ECE (lower = better calibrated)')
    ax.axvline(x=3, color='gray', linestyle='--', alpha=0.4, label='3B threshold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle('Figure 2: Expected Calibration Error (Direct vs CoT)', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig2_ece_scaling.png', bbox_inches='tight')
plt.show()
print('✓ fig2 saved')

## Figure 3 — Reliability Diagrams (for representative models)

In [ ]:
# Show diagrams for 4 key models: 0.5B, 1.5B, 3B, 8B — both modes, gsm8k
representative = ['Qwen2.5-0.5B', 'Qwen2.5-1.5B', 'Qwen2.5-3B', 'Llama-3.1-8B']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))

for col, model_name in enumerate(representative):
    for row, mode in enumerate(['direct', 'cot']):
        ax = axes[row][col]
        key = f'{model_name}/{mode}/gsm8k'
        bins = bin_data.get(key, [])

        if not bins:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        else:
            mids = [b['bin_mid'] for b in bins]
            accs = [b['bin_acc'] for b in bins]
            confs= [b['bin_conf'] for b in bins]

            ax.bar(mids, accs, width=0.08, alpha=0.6, color=COLORS[mode], label='Accuracy')
            ax.plot([0,1], [0,1], 'k--', alpha=0.5, label='Perfect cal.')

        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_title(f'{model_name}\n{mode}', fontsize=9)
        if col == 0:
            ax.set_ylabel('Accuracy')
        if row == 1:
            ax.set_xlabel('Confidence')

fig.suptitle('Figure 3: Reliability Diagrams — GSM8K, 4 representative models', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig3_reliability_diagrams.png', bbox_inches='tight')
plt.show()
print('✓ fig3 saved')

## Figure 4 — Consistency Delta (CoT − Direct) vs Model Size

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, ds in zip(axes, ['gsm8k', 'truthfulqa']):
    subset = delta_df[delta_df.dataset == ds].sort_values('size_b')

    colors = ['#d62728' if d < 0 else '#2ca02c' for d in subset['delta']]
    ax.bar(range(len(subset)), subset['delta'], color=colors, alpha=0.8)
    ax.set_xticks(range(len(subset)))
    ax.set_xticklabels(subset['model'], rotation=30, ha='right', fontsize=8)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_ylabel('Consistency Delta (CoT − Direct)')
    ax.set_title(f'{DATASET_LABELS[ds]}')
    ax.text(0.02, 0.95, 'Green = CoT helps consistency\nRed = CoT hurts consistency',
            transform=ax.transAxes, fontsize=8, va='top')
    ax.grid(True, axis='y', alpha=0.3)

fig.suptitle('Figure 4: Consistency Delta (CoT − Direct) per Model', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig4_consistency_delta.png', bbox_inches='tight')
plt.show()
print('✓ fig4 saved')

## Figure 5 — Combined Trustworthiness Heatmap

In [ ]:
# Build a combined matrix: rows = models (by size), columns = (axis, mode, dataset)
models_ordered = sorted(MODEL_NAMES, key=lambda m: size_map[m])

# Accuracy CoT on GSM8K
acc_cot_gsm = acc_df[(acc_df.mode=='cot') & (acc_df.dataset=='gsm8k')].set_index('model')['accuracy']
acc_dir_gsm = acc_df[(acc_df.mode=='direct') & (acc_df.dataset=='gsm8k')].set_index('model')['accuracy']
acc_delta_gsm = acc_cot_gsm - acc_dir_gsm  # positive = CoT helps accuracy

# ECE CoT - Direct on GSM8K (positive = CoT hurts calibration)
ece_cot_gsm = ece_df[(ece_df.mode=='cot') & (ece_df.dataset=='gsm8k')].set_index('model')['ece']
ece_dir_gsm = ece_df[(ece_df.mode=='direct') & (ece_df.dataset=='gsm8k')].set_index('model')['ece']
ece_delta_gsm = ece_cot_gsm - ece_dir_gsm  # positive = CoT worsens calibration

# Consistency delta (CoT - Direct) GSM8K
con_delta_gsm = delta_df[delta_df.dataset=='gsm8k'].set_index('model')['delta']

heatmap_data = pd.DataFrame({
    'Acc delta\n(CoT-Direct)': acc_delta_gsm,
    'ECE delta\n(CoT-Direct)': ece_delta_gsm,
    'Cons delta\n(CoT-Direct)': con_delta_gsm,
}).reindex(models_ordered)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    heatmap_data,
    annot=True, fmt='.3f',
    cmap='RdYlGn',
    center=0,
    linewidths=0.5,
    ax=ax
)
ax.set_title('Figure 5: CoT Effect on 3 Trustworthiness Axes\n(Green = CoT helps, Red = CoT hurts)',
             fontsize=12)
ax.set_xlabel('Metric')
ax.set_ylabel('Model (sorted by size, top=smallest)')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig5_combined_trustworthiness.png', bbox_inches='tight')
plt.show()
print('✓ fig5 saved')

## Hypothesis Test — Is the main claim confirmed?

In [ ]:
# Main claim: there exists a size threshold below which CoT impairs trustworthiness
# (calibration AND/OR consistency) even if accuracy effects are neutral

print('=== HYPOTHESIS TEST ===\n')
print('H1: CoT worsens calibration (higher ECE) for sub-3B models')
sub3b = ece_df[ece_df.size_b < 3]
h1 = (sub3b[sub3b.mode=='cot']['ece'].mean() > sub3b[sub3b.mode=='direct']['ece'].mean())
print(f'  sub-3B ECE(CoT)={sub3b[sub3b.mode=="cot"]["ece"].mean():.3f}, '
      f'ECE(Direct)={sub3b[sub3b.mode=="direct"]["ece"].mean():.3f}')
print(f'  H1 CONFIRMED: {h1}\n')

print('H2: CoT decreases consistency for sub-3B models')
sub3b_con = delta_df[delta_df.size_b < 3]
h2 = sub3b_con['delta'].mean() < 0
print(f'  sub-3B mean consistency delta (CoT-Direct): {sub3b_con["delta"].mean():.3f}')
print(f'  H2 CONFIRMED: {h2}\n')

print('H3: Pattern reverses at >= 3B (CoT helps both metrics)')
above3b = delta_df[delta_df.size_b >= 3]
ece_above = ece_df[ece_df.size_b >= 3]
h3a = (ece_above[ece_above.mode=='cot']['ece'].mean() < ece_above[ece_above.mode=='direct']['ece'].mean())
h3b = above3b['delta'].mean() > 0
print(f'  >= 3B ECE(CoT) < ECE(Direct): {h3a}')
print(f'  >= 3B consistency delta > 0: {h3b}')
print(f'  H3 CONFIRMED: {h3a and h3b}')

In [ ]:
print('\n✅ STEP 5 COMPLETE — All figures saved to figures/')
print('\nFiles ready for paper:')
for f in sorted(Path(FIGURES_DIR).glob('*.png')):
    print(f'  {f.name}')